# Peak-to-gene association (Trevino et al. 2021)

Spearman correlation between KNN-smoothed accessibility and expression;
significance assessed against GC-bias- and depth-matched background peaks.

In [ ]:
import os, gc
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import muon as mu
import pyranges as pr
from scipy.stats import rankdata
from sklearn.metrics import pairwise_distances

DATA_ROOT = Path(os.environ.get('B_CELL_ATLAS_DATA', 'data'))

## Load MuData

In [ ]:
mdata = mu.read(DATA_ROOT / 'Trevino_multiome_wnn.h5mu')
print(mdata)

## Find peaks within 10 kb of marker genes

In [ ]:
marker_genes = ['EOMES', 'PPP1R17', 'NEUROG1', 'NEUROD2', 'TBR1', 'BCL11B']

# gene coordinates stored in rna.var from preprocessing
genes_df = mdata['rna'].var[['Chromosome', 'Start', 'End', 'GeneName', 'GeneID']].dropna()
marker_df = genes_df[genes_df['GeneName'].isin(marker_genes)].copy()
marker_df['Start'] = marker_df['Start'].astype(int)
marker_df['End'] = marker_df['End'].astype(int)

atac_var = mdata['atac'].var.copy()

def proximal_peaks(gene_row, atac_var, window=10000):
    """Return peak IDs within `window` bp of a gene body."""
    gene_gr = pr.PyRanges(chromosomes=[gene_row['Chromosome']],
                          starts=[gene_row['Start'] - window],
                          ends=[gene_row['End'] + window])
    peaks_gr = pr.PyRanges(chromosomes=atac_var['seqnames'],
                           starts=atac_var['start'],
                           ends=atac_var['end'])
    overlap = peaks_gr.overlap(gene_gr)
    if len(overlap) == 0:
        return []
    peak_strings = overlap.Chromosome.astype(str) + ':' + overlap.Start.astype(str) + '-' + overlap.End.astype(str)
    return list(atac_var.index[atac_var['seqnames'].astype(str) + ':' + atac_var['start'].astype(str) + '-' + atac_var['end'].astype(str)
                               .isin(peak_strings)])

marker_df['peak_ids'] = marker_df.apply(lambda r: proximal_peaks(r, atac_var), axis=1)
marker_df[['GeneName', 'peak_ids']]

## KNN smoothing

In [ ]:
def knn_smooth(X, connectivities):
    """Replace each cell profile with the mean of its KNN neighbors."""
    X_smooth = scipy.sparse.csr_matrix.dot(connectivities.T, X).astype(np.float32)
    if scipy.sparse.issparse(X_smooth):
        X_smooth = X_smooth.toarray()
    col_sums = np.array(X.sum(1)).ravel()
    col_sums[col_sums == 0] = 1
    X_smooth = X_smooth / col_sums.reshape(-1, 1)
    X_smooth = np.nan_to_num(X_smooth)
    return X_smooth

conn = mdata.obsp['connectivities']
all_peaks = list(set(p for peaks in marker_df['peak_ids'] for p in peaks))
sdata_atac = mdata['atac'][:, all_peaks].copy()
sdata_rna  = mdata['rna'][:, marker_df['GeneName'].tolist()].copy()
sdata_atac.layers['smooth'] = knn_smooth(sdata_atac.X, conn)
sdata_rna.layers['smooth']  = knn_smooth(sdata_rna.X, conn)

## Compute peak–gene correlation with GC-matched background test

In [ ]:
sc.pp.calculate_qc_metrics(mdata['atac'], percent_top=None, log1p=False, inplace=True)
mdata['atac'].var['gc_bin']    = pd.cut(mdata['atac'].var['GCbias'],
                                         bins=np.arange(0, 1, 0.05))
mdata['atac'].var['count_bin'] = pd.cut(mdata['atac'].var['n_cells_by_counts'],
                                         bins=np.arange(0, mdata['atac'].var['n_cells_by_counts'].max(), 100))

N_BG = 30
results = []

for _, row in marker_df.iterrows():
    gene = row['GeneName']
    peaks = row['peak_ids']
    if not peaks:
        continue
    gex = sdata_rna[:, gene].layers['smooth'].ravel()
    acc = sdata_atac[:, peaks].layers['smooth']

    # Spearman correlations between each peak and the gene
    cors = 1 - np.ravel(
        pairwise_distances(np.apply_along_axis(rankdata, 1, acc.T),
                           rankdata(gex).reshape(1, -1),
                           metric='correlation'))

    for peak, cor in zip(peaks, cors):
        gc_bin    = mdata['atac'].var.loc[peak, 'gc_bin']
        cnt_bin   = mdata['atac'].var.loc[peak, 'count_bin']
        bg_mask   = ((mdata['atac'].var['gc_bin'] == gc_bin) &
                     (mdata['atac'].var['count_bin'] == cnt_bin))
        bg_peaks  = mdata['atac'].var_names[bg_mask]
        replace   = len(bg_peaks) < N_BG
        bg_sample = np.random.choice(bg_peaks, N_BG, replace=replace)
        bg_acc    = knn_smooth(mdata['atac'][:, bg_sample].X, conn)
        bg_cors   = 1 - np.ravel(
            pairwise_distances(np.apply_along_axis(rankdata, 1, bg_acc.T),
                               rankdata(gex).reshape(1, -1),
                               metric='correlation'))
        pval = 1 - scipy.stats.norm.cdf(cor, bg_cors.mean(), bg_cors.std() + 1e-10)
        results.append({'gene': gene, 'peak': peak, 'cor': cor, 'pval': pval})
    gc.collect()

res = pd.DataFrame(results)
# Benjamini-Hochberg FDR
ranked = rankdata(res['pval'])
res['fdr'] = (res['pval'] * len(res)) / ranked
res['fdr'] = res['fdr'].clip(upper=1)
res.head()

## Results

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(res['cor'], -np.log10(res['fdr'] + 1e-10), s=10, alpha=0.6)
plt.axhline(-np.log10(0.1), color='red', linestyle='--', linewidth=0.8)
plt.xlabel('Spearman R')
plt.ylabel('-log10(FDR)')
plt.title('Peak–gene correlation')
plt.tight_layout()
plt.show()

In [ ]:
sig = res[res['fdr'] < 0.1]
summary = (sig.groupby('gene')
           .agg(n_sig_peaks=('peak', 'count'))
           .reset_index()
           .merge(res.groupby('gene').size().reset_index(name='n_tested')))
summary['frac'] = summary['n_sig_peaks'] / summary['n_tested']
sns.barplot(data=summary, x='gene', y='frac')
plt.ylabel('Fraction significant peaks (FDR < 0.1)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Save

In [ ]:
res.to_csv(DATA_ROOT / 'peak2gene_correlations.csv', index=False)